# Рівень 2. Згорткова нейромережа (CNN), навчена з нуля

**Задача:** та сама, що на рівні 1 — за ASCII-рисунком визначити об'єкт (10 класів).
**Метод рівня 2:** Deep Learning — згорткова нейромережа (CNN), яку ми навчаємо з нуля на наших даних.

**Що змінюється порівняно з рівнем 1.** На рівні 1 ознаки придумували ми (розміри, частоти символів, HOG), а класифікатор лише обирав серед них. Нейромережа отримує «сирий» рисунок і **сама вчиться**, які візерунки на ньому корисні для розпізнавання. Висновки рівня 1 показали, що впираємося саме в ручні ознаки, тому це логічний наступний крок.

**Що залишається тим самим** (щоб порівняння було чесним):
- ті самі 1341 рисунок і **те саме розбиття** train / val / test з `splits.csv`, створене на рівні 1;
- ті самі метрики (macro-F1 як основна, accuracy), той самий спосіб вимірювання часу і пам'яті.

**Залізо.** Рівень 1 рахувався на CPU ноутбука (Intel Core i7-8850H). Нейромережі на практиці навчають на GPU, тому фінальний запуск рівня 2 зроблено в Google Colab на GPU. Залізо кожного рівня вказано в порівняльній таблиці: потреба в GPU — теж частина «ціни» методу.

План:
1. Дані та розбиття
2. Як влаштована CNN і як вона навчається
3. Спроба 1: CNN на зображеннях 128×128
4. Чому зображення не спрацювали
5. Спроба 2: CNN на сітці символів
6. Вибір моделі та фінальна оцінка на test
7. Порівняння з рівнем 1
8. Висновки

> **Як запустити.** У Google Colab: Runtime → Change runtime type → GPU, потім Runtime → Run all; повний прогін триває кілька хвилин. Локально на CPU ноутбука — ≈ 15 хвилин. Майже весь час займає навчання двох нейромереж.

## 1. Дані та розбиття

Якщо ноутбук запущено в Google Colab, спочатку завантажуємо репозиторій з датасетом і `splits.csv` з GitHub. Локально цей крок пропускається — дані вже поруч.

In [ ]:
import os
import subprocess
import sys

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/ML'):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/seibtoy/ML.git', '/content/ML'], check=True)
    os.chdir('/content/ML/level2')
print('Робоча папка:', os.getcwd())

In [ ]:
import copy
import json
import platform
import string
import threading
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score

SEED = 42
DATASET_DIR = '../dataset'
SPLITS_PATH = '../splits.csv'
IMAGE_SIZE = 128
BATCH_SIZE = 32

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def cpu_name():
    if sys.platform == 'darwin':
        return subprocess.run(['sysctl', '-n', 'machdep.cpu.brand_string'], capture_output=True, text=True).stdout.strip()
    with open('/proc/cpuinfo') as f:
        return next(line.split(':', 1)[1].strip() for line in f if line.startswith('model name'))


hardware = f'GPU {torch.cuda.get_device_name(0)}' if device.type == 'cuda' else f'CPU {cpu_name()}'
print('Залізо:', hardware)
print('Python', platform.python_version())
for module in [torch, np, pd, matplotlib, psutil]:
    print(module.__name__, module.__version__)
print('scikit-learn', __import__('sklearn').__version__, '| pillow', Image.__version__)

In [ ]:
splits = pd.read_csv(SPLITS_PATH)
texts = [open(os.path.join(DATASET_DIR, f), encoding='utf-8').read() for f in splits['file']]
classes = sorted(splits['label'].unique())
labels = splits['label'].map({c: i for i, c in enumerate(classes)}).to_numpy()

parts = {name: np.flatnonzero(splits['split'] == name) for name in ['train', 'val', 'test']}
y = {name: labels[idx] for name, idx in parts.items()}
{name: len(idx) for name, idx in parts.items()}

Розбиття не створюється заново, а читається з `splits.csv` — тому test тут ті самі 202 рисунки, на яких оцінювався рівень 1.

Клас `Measure` — той самий, що на рівні 1 (час виконання блоку коду і пікова RAM процесу), з двома доповненнями для GPU:
- **пікова VRAM** — `torch.cuda.max_memory_allocated()`: скільки пам'яті відеокарти зайняли тензори PyTorch;
- **синхронізація** — GPU виконує обчислення асинхронно: Python може піти далі, поки відеокарта ще рахує. Тому перед зупинкою таймера чекаємо завершення всіх обчислень (`torch.cuda.synchronize()`), інакше час був би заниженим.

In [ ]:
class Measure:
    def __enter__(self):
        self.process = psutil.Process()
        self.peak_rss = self.process.memory_info().rss
        self.running = True
        self.sampler = threading.Thread(target=self.sample, daemon=True)
        self.sampler.start()
        if device.type == 'cuda':
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        self.started = time.perf_counter()
        return self

    def sample(self):
        while self.running:
            self.peak_rss = max(self.peak_rss, self.process.memory_info().rss)
            time.sleep(0.005)

    def __exit__(self, *exc):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        self.seconds = time.perf_counter() - self.started
        self.running = False
        self.sampler.join()
        self.peak_rss = max(self.peak_rss, self.process.memory_info().rss)
        self.peak_mb = self.peak_rss / 2**20
        self.peak_vram_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None

## 2. Як влаштована CNN і як вона навчається

### Будова мережі

Мережа — це послідовність **шарів**. Кожен шар перетворює вхідні числа на нові числа за допомогою **ваг** — параметрів, які підбираються під час навчання.

- **Згортковий шар (`Conv2d`)** — набір маленьких фільтрів 3×3, які «ковзають» по зображенню і в кожному місці перевіряють, чи є там певний візерунок (штрих, кут, дуга). Значення фільтрів не задаються вручну — вони і є вагами, яких мережа навчається. Перші шари вчаться знаходити прості штрихи (схоже на HOG з рівня 1, але вивчене, а не задане формулою), наступні — комбінації штрихів, ще глибші — частини об'єктів.
- **BatchNorm** — нормалізує значення між шарами, щоб навчання було стабільнішим.
- **ReLU** — проста нелінійна функція `max(0, x)`. Без нелінійності кілька шарів поспіль були б еквівалентні одному.
- **MaxPool 2×2** — зменшує зображення вдвічі, залишаючи найсильніший сигнал у кожному квадраті 2×2. Завдяки цьому глибші шари «бачать» більшу частину рисунка.
- **Global pooling** — усереднює кожну карту ознак по всьому зображенню; результат — вектор ознак фіксованої довжини, незалежно від розміру входу.
- **Dropout** — під час навчання випадково «вимикає» 30% ознак, щоб мережа не покладалася на окремі з них (захист від перенавчання).
- **Linear** — фінальний шар: з вектора ознак рахує 10 чисел, по одному на клас. Найбільше число — відповідь мережі.

Чотири блоки «Conv → BatchNorm → ReLU → MaxPool» поспіль — це і є «глибока» частина мережі: кожен блок будує ознаки з ознак попереднього.

### Навчання

1. **Батч (batch)** — мережа отримує по 32 рисунки за раз.
2. Рахуємо **loss** (функція втрат, cross-entropy) — число, яке показує, наскільки відповіді мережі на цьому батчі неправильні. Як і `class_weight='balanced'` на рівні 1, помилки на рідкісних класах важать більше.
3. **Зворотне поширення (backpropagation)** визначає для кожної ваги, в який бік її трохи змінити, щоб loss зменшився.
4. **Оптимізатор AdamW** робить цей маленький крок для всіх ваг. Розмір кроку — **learning rate**. Він змінюється за розкладом One-Cycle: спочатку зростає, потім плавно спадає до нуля.
5. **Епоха** — один прохід по всіх 938 рисунках train. Повторюємо десятки епох.

Після кожної епохи рахуємо macro-F1 на **validation** і запам'ятовуємо версію мережі з найкращим значенням. У кінці повертаємося саме до неї. Так ми обираємо момент зупинки за validation (аналог early stopping), а test, як і на рівні 1, не використовуємо до фінальної оцінки.

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def predict_logits(model, x):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(x[i:i + 256].to(device)) for i in range(0, len(x), 256)])


def train(model, x_train, x_val, epochs, lr, augment=None):
    model.to(device)
    x_train, x_val = x_train.to(device), x_val.to(device)
    generator = torch.Generator().manual_seed(SEED)
    counts = np.bincount(y['train'], minlength=len(classes))
    class_weights = torch.tensor(len(y['train']) / (len(classes) * counts), dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs, pct_start=0.15)
    y_train, y_val = torch.from_numpy(y['train']).to(device), torch.from_numpy(y['val']).to(device)

    history, best_f1, best_state = [], -1, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x_train), generator=generator).to(device)
        train_loss, train_pred = 0.0, torch.empty(len(x_train), dtype=torch.long, device=device)
        for start in range(0, len(order), BATCH_SIZE):
            batch = order[start:start + BATCH_SIZE]
            xb = augment(x_train[batch], generator) if augment else x_train[batch]
            logits = model(xb)
            loss = loss_fn(logits, y_train[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(batch)
            train_pred[batch] = logits.argmax(1)
        scheduler.step()

        val_logits = predict_logits(model, x_val)
        val_f1 = f1_score(y['val'], val_logits.argmax(1).cpu().numpy(), average='macro')
        history.append({
            'epoch': epoch + 1,
            'train loss': train_loss / len(x_train),
            'val loss': loss_fn(val_logits, y_val).item(),
            'train macro-F1': f1_score(y['train'], train_pred.cpu().numpy(), average='macro'),
            'val macro-F1': val_f1,
        })
        if val_f1 > best_f1:
            best_f1, best_state = val_f1, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return pd.DataFrame(history)


def plot_history(history, title):
    best = history.loc[history['val macro-F1'].idxmax()]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    history.plot(x='epoch', y=['train loss', 'val loss'], ax=axes[0], title='Loss (менше — краще)')
    history.plot(x='epoch', y=['train macro-F1', 'val macro-F1'], ax=axes[1], title='Macro-F1 (більше — краще)')
    for ax in axes:
        ax.axvline(best['epoch'], color='gray', linestyle='--', linewidth=1)
    fig.suptitle(f"{title}: найкраща епоха {int(best['epoch'])}, val macro-F1 = {best['val macro-F1']:.3f}")
    plt.tight_layout()
    plt.show()

Примітка до графіків: **train macro-F1 рахується по ходу епохи** на тих самих батчах, на яких мережа вчиться (з dropout і, у спробі 2, з аугментацією). Тому він трохи занижений порівняно з оцінкою «чистої» мережі на train, але добре показує тенденцію. Окремий прохід по всьому train після кожної епохи подовжив би навчання приблизно на третину.

## 3. Спроба 1: CNN на зображеннях 128×128

Найпряміший варіант для напрямку «зображення»: беремо ту саму функцію `render()` з рівня 1 (текст → картинка 128×128) і подаємо картинки в CNN з чотирма згортковими блоками (16 → 32 → 64 → 128 фільтрів).

In [ ]:
FONT = ImageFont.truetype(os.path.join(matplotlib.get_data_path(), 'fonts/ttf/DejaVuSansMono.ttf'), 16)
CHAR_WIDTH = FONT.getlength('M')
LINE_HEIGHT = 19
MARGIN = 8


def render(text, size=IMAGE_SIZE):
    lines = text.rstrip('\n').split('\n')
    width = int(np.ceil(max(len(line) for line in lines) * CHAR_WIDTH))
    height = len(lines) * LINE_HEIGHT
    side = max(width, height) + 2 * MARGIN
    canvas = Image.new('L', (side, side), 255)
    draw = ImageDraw.Draw(canvas)
    left, top = (side - width) // 2, (side - height) // 2
    for i, line in enumerate(lines):
        draw.text((left, top + i * LINE_HEIGHT), line, fill=0, font=FONT)
    canvas = canvas.resize((size, size), Image.Resampling.LANCZOS)
    return 1 - np.asarray(canvas, dtype=np.float32) / 255


def to_images(texts):
    return torch.from_numpy(np.stack([render(t) for t in texts])).unsqueeze(1)


class ImageCNN(nn.Module):
    def __init__(self, widths=(16, 32, 64, 128)):
        super().__init__()
        blocks, channels = [], 1
        for width in widths:
            blocks.append(conv_block(channels, width))
            channels = width
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(channels, len(classes)))

    def forward(self, x):
        return self.head(self.features(x))

In [ ]:
images, image_prep_cost = {}, {}
for name, idx in parts.items():
    with Measure() as m:
        images[name] = to_images([texts[i] for i in idx])
    image_prep_cost[name] = m

torch.manual_seed(SEED)
image_model = ImageCNN()
print('Параметрів (ваг) у мережі:', count_parameters(image_model))

with Measure() as image_fit_cost:
    image_history = train(image_model, images['train'], images['val'], epochs=40, lr=3e-3)
print(f'Навчання: {image_fit_cost.seconds / 60:.1f} хв, пік RAM {image_fit_cost.peak_mb:.0f} MB, пік VRAM {image_fit_cost.peak_vram_mb} MB')

In [ ]:
plot_history(image_history, 'Спроба 1, CNN на зображеннях')

## 4. Чому зображення не спрацювали

Результат спроби 1 **гірший за рівень 1** (val macro-F1 рівня 1 — 0.485). Під час підготовки ми також пробували цю мережу з аугментацією і ширшу версію (32 → 256 фільтрів) — вони не дали кращого результату.

Графіки показують, що мережа запам'ятовує train (train F1 росте), а validation застрягає. Причина — у самому представленні даних. Рисунок шириною 100+ символів, стиснутий до 128 пікселів, отримує приблизно 1 піксель на символ: `/`, `|`, `o` і `#` перетворюються на однакові сірі плями. Рівень 1 частково обходив це, бо частоти символів рахував прямо з тексту. CNN на картинці бачить лише розмитий силует.

In [ ]:
widest = int(np.argmax([max(len(line) for line in t.split('\n')) for t in texts]))
full = render(texts[widest], size=1024)
rows, cols = np.nonzero(full > 0.3)
center_row, center_col = int(rows.mean()), int(cols.mean())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(full, cmap='gray_r')
axes[0].set_title('Рисунок у високій роздільності')
axes[1].imshow(render(texts[widest]), cmap='gray_r')
axes[1].set_title('Те, що бачить CNN: 128×128')
axes[2].imshow(full[center_row - 80:center_row + 80, center_col - 120:center_col + 120], cmap='gray_r')
axes[2].set_title('Фрагмент у високій роздільності: символи читаються')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Спроба 2: CNN на сітці символів

**Ідея.** ASCII-арт — це вже двовимірне зображення, тільки «пікселі» в ньому — символи, а не яскравість. Тож замість того, щоб малювати текст і стискати картинку, подаємо в мережу саму **сітку символів** 64×112 (найбільший рисунок у датасеті — 60 рядків × 109 символів):
- кожен символ замінюємо його номером у словнику (95 друкованих символів + «інший»); пробіл і порожнє місце — 0;
- рисунок розміщуємо по центру сітки.

**Embedding.** Номер символу сам по собі нічого не означає: `/` і `\` просто отримують різні номери в словнику, і з того, що один номер більший за інший, не випливає нічого корисного. Тому перший шар мережі — **embedding**: таблиця, де кожному символу відповідає вектор із кількох чисел. Ці вектори теж вчаться під час навчання, і мережа сама вирішує, які символи «схожі» між собою (наприклад, `-` і `_`). Після embedding сітка стає звичайним багатоканальним «зображенням» розміру 64×112, і далі працюють ті самі згорткові блоки, що й у спробі 1.

Так зберігаються і форма рисунка (як у зображенні), і точні символи (як у частотах символів на рівні 1), без втрат від стиснення.

**Аугментація** — штучне розширення train, щоб мережа менше запам'ятовувала конкретні рисунки. На кожному батчі рисунок випадково:
- **зсувається** в межах сітки (об'єкт той самий, просто в іншому місці);
- з імовірністю 50% **віддзеркалюється** по горизонталі. При цьому несиметричні символи замінюються на дзеркальні: `/` ↔ `\`, `(` ↔ `)`, `<` ↔ `>`, `[` ↔ `]`, `b` ↔ `d` і т. д. — інакше віддзеркалений кіт складався б із «неправильних» штрихів.

Validation і test не аугментуються.

In [ ]:
GRID_HEIGHT, GRID_WIDTH = 64, 112
VOCAB = string.printable[:94]
CHAR_INDEX = {ch: i + 1 for i, ch in enumerate(VOCAB)}
OTHER_INDEX = len(VOCAB) + 1
MIRROR = dict(zip('/\\()<>[]{}bdpq', '\\/)(><][}{dbqp'))

mirror_index = torch.arange(OTHER_INDEX + 1)
for a, b in MIRROR.items():
    mirror_index[CHAR_INDEX[a]] = CHAR_INDEX[b]


def to_grid(text):
    lines = text.rstrip('\n').split('\n')
    grid = np.zeros((GRID_HEIGHT, GRID_WIDTH), dtype=np.int64)
    top = (GRID_HEIGHT - len(lines)) // 2
    left = (GRID_WIDTH - max(len(line) for line in lines)) // 2
    for r, line in enumerate(lines):
        for c, ch in enumerate(line):
            if not ch.isspace():
                grid[top + r, left + c] = CHAR_INDEX.get(ch, OTHER_INDEX)
    return grid


def to_grids(texts):
    return torch.from_numpy(np.stack([to_grid(t) for t in texts]))


def mirror(grids):
    return mirror_index.to(grids.device)[grids.flip(-1)]


def augment_grids(grids, generator):
    grids = grids.clone()
    flip = (torch.rand(len(grids), generator=generator) < 0.5).to(grids.device)
    grids[flip] = mirror(grids[flip])
    for i, grid in enumerate(grids):
        rows = grid.any(1).nonzero().flatten()
        cols = grid.any(0).nonzero().flatten()
        dr = torch.randint(-rows.min().item(), GRID_HEIGHT - rows.max().item(), (1,), generator=generator).item()
        dc = torch.randint(-cols.min().item(), GRID_WIDTH - cols.max().item(), (1,), generator=generator).item()
        grids[i] = grid.roll((dr, dc), (0, 1))
    return grids


class GridCNN(nn.Module):
    def __init__(self, widths, embedding_dim):
        super().__init__()
        self.embed = nn.Embedding(OTHER_INDEX + 1, embedding_dim, padding_idx=0)
        blocks, channels = [], embedding_dim
        for width in widths:
            blocks.append(conv_block(channels, width))
            channels = width
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * channels, len(classes)))

    def forward(self, grid):
        x = self.features(self.embed(grid).permute(0, 3, 1, 2))
        return self.head(torch.cat([x.mean((2, 3)), x.amax((2, 3))], dim=1))

Наприкінці мережа об'єднує два види global pooling: **середнє** (наскільки ознака присутня в рисунку загалом) і **максимум** (чи є вона хоча б в одному місці). Друге корисне для дрібних деталей — наприклад, характерних очей `o o` чи вусів `=`.

Подивимося, як виглядає аугментація на одному рисунку з train:

In [ ]:
grids, grid_prep_cost = {}, {}
for name, idx in parts.items():
    with Measure() as m:
        grids[name] = to_grids([texts[i] for i in idx])
    grid_prep_cost[name] = m

INDEX_CHAR = {i: ch for ch, i in CHAR_INDEX.items()}


def grid_to_text(grid):
    rows = [''.join(INDEX_CHAR.get(int(v), '?') if v else ' ' for v in row).rstrip() for row in grid]
    nonempty = [i for i, row in enumerate(rows) if row]
    return '\n'.join(rows[nonempty[0]:nonempty[-1] + 1])


sample = grids['train'][[i for i in range(len(y['train'])) if classes[y['train'][i]] == 'cat'][5]]
print('Оригінал:\n' + grid_to_text(sample))
print('\nВіддзеркалення (несиметричні символи замінено на дзеркальні):\n' + grid_to_text(mirror(sample[None])[0]))
print('\nВипадкова аугментація, як під час навчання (зсув і, можливо, віддзеркалення):\n' + grid_to_text(augment_grids(sample[None], torch.Generator().manual_seed(1))[0]))

In [ ]:
GRID_WIDTHS = (32, 64, 128)
GRID_EMBEDDING = 16
GRID_EPOCHS = 80
GRID_LR = 3e-3

torch.manual_seed(SEED)
grid_model = GridCNN(GRID_WIDTHS, GRID_EMBEDDING)
print('Параметрів (ваг) у мережі:', count_parameters(grid_model))

with Measure() as grid_fit_cost:
    grid_history = train(grid_model, grids['train'], grids['val'], epochs=GRID_EPOCHS, lr=GRID_LR, augment=augment_grids)
print(f'Навчання: {grid_fit_cost.seconds / 60:.1f} хв, пік RAM {grid_fit_cost.peak_mb:.0f} MB, пік VRAM {grid_fit_cost.peak_vram_mb} MB')

In [ ]:
plot_history(grid_history, 'Спроба 2, CNN на сітці символів')

**Розмір мережі.** Під час підготовки ми порівняли цю мережу (32 → 64 → 128 фільтрів, embedding 16, ≈ 100 тис. ваг) з ширшою версією (64 → 128 → 256 фільтрів, embedding 32, ≈ 400 тис. ваг). Ширша дала на validation майже той самий результат (різниця ≈ 0.01–0.02 macro-F1, у межах випадкових коливань на 201 рисунку), але навчалася ≈ 2.5 раза довше і сильніше перенавчалася (train macro-F1 ≈ 0.97). Тому обрано меншу мережу: та сама якість за меншу ціну.

## 6. Вибір моделі та фінальна оцінка на test

Як і на рівні 1, модель обираємо **за validation**: та спроба, у якої найкращий val macro-F1, іде на test. Test використовується один раз.

In [ ]:
attempts = {
    'Спроба 1: CNN на зображеннях': (image_model, image_history, images, image_prep_cost, image_fit_cost),
    'Спроба 2: CNN на сітці символів': (grid_model, grid_history, grids, grid_prep_cost, grid_fit_cost),
}
summary = pd.DataFrame([
    {
        'спроба': name,
        'параметрів': count_parameters(model),
        'епох': len(history),
        'найкраща епоха': int(history.loc[history['val macro-F1'].idxmax(), 'epoch']),
        'val macro-F1': history['val macro-F1'].max(),
        'час навчання, хв': fit_cost.seconds / 60,
    }
    for name, (model, history, _, _, fit_cost) in attempts.items()
])
best_name = summary.loc[summary['val macro-F1'].idxmax(), 'спроба']
print('Обрано за validation:', best_name)
summary.round(3)

In [ ]:
final_model, final_history, final_inputs, final_prep_cost, final_fit_cost = attempts[best_name]

with Measure() as predict_cost:
    test_pred = predict_logits(final_model, final_inputs['test']).argmax(1).cpu().numpy()

train_seconds = final_prep_cost['train'].seconds + final_fit_cost.seconds
inference_seconds = final_prep_cost['test'].seconds + predict_cost.seconds

test_accuracy = accuracy_score(y['test'], test_pred)
test_macro_f1 = f1_score(y['test'], test_pred, average='macro')

print(f'Модель: {best_name}')
print(f'Test accuracy: {test_accuracy:.3f}')
print(f'Test macro-F1: {test_macro_f1:.3f}')
print()
print(classification_report(y['test'], test_pred, target_names=classes, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    [classes[i] for i in y['test']], [classes[i] for i in test_pred],
    labels=classes, xticks_rotation=45, colorbar=False, ax=ax,
)
ax.set_title('Матриця помилок на test')
ax.set_xlabel('передбачений клас')
ax.set_ylabel('справжній клас')
plt.show()

In [ ]:
wrong = np.flatnonzero(test_pred != y['test'])
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.flat, wrong[:10]):
    ax.imshow(render(texts[parts['test'][i]]), cmap='gray_r')
    ax.set_title(f"справжній: {classes[y['test'][i]]}\nмодель: {classes[test_pred[i]]}", fontsize=10)
    ax.axis('off')
plt.suptitle('Приклади помилок на test')
plt.tight_layout()
plt.show()

**Час і пам'ять** рахуються так само, як на рівні 1:
- **навчання** = підготовка входів train + навчання мережі. Навчання включає оцінку на validation після кожної епохи — вона є частиною процедури (саме за нею обирається найкраща епоха);
- **інференс** = підготовка входів test + прохід мережі по test;
- **RAM** — пік процесу; **VRAM** — пік пам'яті GPU (якщо запуск на GPU).

In [ ]:
final_costs = [*final_prep_cost.values(), final_fit_cost, predict_cost]
vram = [m.peak_vram_mb for m in final_costs if m.peak_vram_mb is not None]

results = {
    'level': 2,
    'method': f'CNN з нуля ({best_name.split(": ")[1]})',
    'hardware': hardware,
    'test_accuracy': round(test_accuracy, 4),
    'test_macro_f1': round(test_macro_f1, 4),
    'train_seconds': round(train_seconds, 3),
    'inference_seconds': round(inference_seconds, 3),
    'inference_ms_per_sample': round(1000 * inference_seconds / len(parts['test']), 3),
    'peak_ram_mb': round(max(m.peak_mb for m in final_costs)),
    'peak_vram_mb': round(max(vram)) if vram else None,
}
with open('results.json', 'w') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(json.dumps(results, ensure_ascii=False, indent=2))

pd.DataFrame([
    ['Залізо', hardware],
    ['Метрика на test: accuracy', f'{test_accuracy:.3f}'],
    ['Метрика на test: macro-F1', f'{test_macro_f1:.3f}'],
    ['Час навчання: підготовка train', f"{final_prep_cost['train'].seconds:.2f} с"],
    ['Час навчання: навчання мережі', f'{final_fit_cost.seconds:.1f} с ({final_fit_cost.seconds / 60:.1f} хв)'],
    ['Час навчання: разом', f'{train_seconds:.1f} с'],
    ['Час інференсу: підготовка test', f"{final_prep_cost['test'].seconds:.3f} с"],
    ['Час інференсу: прохід мережі', f'{predict_cost.seconds:.3f} с'],
    ['Час інференсу: разом', f'{inference_seconds:.3f} с ({results["inference_ms_per_sample"]:.2f} мс на рисунок)'],
    ['Пікова RAM процесу', f"{results['peak_ram_mb']} MB"],
    ['Пікова VRAM', f"{results['peak_vram_mb']} MB" if vram else 'не використовується (CPU)'],
], columns=['Показник', 'Значення'])

## 7. Порівняння з рівнем 1

Обидва рівні оцінені на тих самих 202 рисунках test. Залізо різне, і воно вказане в таблиці: час нейромережі на GPU не можна напряму порівнювати з часом на CPU, але саме так кожен із цих методів використовують на практиці — класичному ML вистачає звичайного процесора, нейромережу навчають на GPU.

In [ ]:
with open('../level1/results.json') as f:
    level1 = json.load(f)
level1.setdefault('hardware', 'CPU Intel(R) Core(TM) i7-8850H CPU @ 2.60GHz')

comparison = pd.DataFrame([level1, results]).set_index('level')
comparison = comparison[['method', 'hardware', 'test_accuracy', 'test_macro_f1', 'train_seconds', 'inference_ms_per_sample', 'peak_ram_mb', 'peak_vram_mb']]
comparison.columns = ['метод', 'залізо', 'test accuracy', 'test macro-F1', 'навчання, с', 'інференс, мс/рисунок', 'пік RAM, MB', 'пік VRAM, MB']
comparison

## 8. Висновки

_(буде заповнено після запуску в Colab)_